# Logistic Regression Classification — Compact Notebook

This notebook implements a minimal, step-by-step Logistic Regression classification example.

**For comprehensive explanations, theory, and detailed examples**, see:
**[`../teaching/01_logistic_regression.md`](../teaching/01_logistic_regression.md)**

## Quick Steps Overview:

- **Step 1** — Import libraries & load data  
- **Step 2** — Read data and choose features & target  
- **Step 3** — Exploratory Data Analysis (EDA)  
- **Step 4** — Data cleaning and preparation  
- **Step 5** — Split data (Train / Test) and scale features  
- **Step 6** — Train Logistic Regression model  
- **Step 7** — Make predictions and evaluate performance  
- **Step 8** — Visualize decision boundary and results  
- **Step 9** — Model interpretation and coefficients analysis  
- **Step 10** — Compare with other classification models

---

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations |
| `pandas` | Loading and analysing the dataset |
| `matplotlib` | Plotting the decision boundary |
| `seaborn` | Statistical visualisations |

In [ ]:
# Import basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("Ready for logistic regression classification!")

## Step 2: Load the Dataset

We load the Social Network Ads dataset: 400 users with `Age`, `EstimatedSalary`, and `Purchased` (0 or 1).

We select only columns 2 and 3 (Age and EstimatedSalary) as features. This 2-feature setup makes it possible to visualise the decision boundary in 2D — an important pedagogical choice.

In [ ]:
# Load the dataset
dataset = pd.read_csv('../data/social_network_ads.csv')

print("Dataset loaded successfully!")
print(f"Dataset shape: {dataset.shape}")

# Display basic information
print(f"\nDataset head:")
print(dataset.head())

print(f"\nDataset info:")
print(dataset.info())

# Define features and target
X = dataset.iloc[:, :-1].values  # All columns except last (Age, EstimatedSalary)
y = dataset.iloc[:, -1].values   # Last column (Purchased - target)

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target classes: {np.unique(y)}")

## Step 3: Exploratory Data Analysis (EDA)

Before fitting any model, we need to understand the data:

- **Class balance:** Is the dataset roughly 50/50 between purchased and not purchased? Severe imbalance would make accuracy misleading and require resampling or class weights.
- **Feature ranges:** Age and salary are on different scales — a sign that feature scaling will be needed.
- **Visual separation:** A scatter plot of age vs salary, coloured by purchase decision, will show whether the classes are linearly separable — i.e., whether a straight line can divide buyers from non-buyers.

In [ ]:
# Basic dataset statistics
print("Dataset Description:")
print(dataset.describe())

print("\nMissing values:")
print(dataset.isnull().sum())

print("\nTarget distribution:")
print(dataset['Purchased'].value_counts())
print("\nTarget proportions:")
print(dataset['Purchased'].value_counts(normalize=True))

# Visualize the data distribution
plt.figure(figsize=(15, 5))

# Age distribution by target
plt.subplot(1, 3, 1)
for target in [0, 1]:
    subset = dataset[dataset['Purchased'] == target]
    plt.hist(subset['Age'], alpha=0.7, label=f'Purchased={target}', bins=20)
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Age Distribution by Purchase Decision')
plt.legend()

# Salary distribution by target
plt.subplot(1, 3, 2)
for target in [0, 1]:
    subset = dataset[dataset['Purchased'] == target]
    plt.hist(subset['EstimatedSalary'], alpha=0.7, label=f'Purchased={target}', bins=20)
plt.xlabel('Estimated Salary')
plt.ylabel('Frequency')
plt.title('Salary Distribution by Purchase Decision')
plt.legend()

# Scatter plot
plt.subplot(1, 3, 3)
colors = ['red', 'blue']
for i, target in enumerate([0, 1]):
    subset = dataset[dataset['Purchased'] == target]
    plt.scatter(subset['Age'], subset['EstimatedSalary'], 
               c=colors[i], alpha=0.6, label=f'Purchased={target}')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.title('Age vs Salary by Purchase Decision')
plt.legend()

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Binary classification problem (0: Not Purchased, 1: Purchased)")
print("- Features: Age and Estimated Salary (both continuous)")
print("- Need to check for linear separability")

## Step 4: Data Cleaning

The Social Network Ads dataset is pre-cleaned — no missing values or categorical variables.

In a real project, this step would handle: missing value imputation, outlier treatment, and removing irrelevant columns. Even clean datasets benefit from a quick `dataset.isnull().sum()` and `dataset.describe()` check.

In [ ]:
# Check data quality
print("Missing values per column:")
print(dataset.isnull().sum())

print("\nData types:")
print(dataset.dtypes)

print("\nDataset shape:", dataset.shape)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature ranges:")
print(f"Age: {X[:, 0].min():.0f} to {X[:, 0].max():.0f}")
print(f"Salary: ${X[:, 1].min():,.0f} to ${X[:, 1].max():,.0f}")

print("\nData is clean and ready for logistic regression!")
print("Note: Feature scaling will be applied in next step")

## Step 5: Train/Test Split and Feature Scaling

**Why split before scaling?**
The scaler learns the mean and standard deviation from the data it is fit on. If we scaled the full dataset before splitting, the test set's statistics would influence the scaler — this is data leakage. The scaler must see only training data.

**Why scaling for Logistic Regression?**
Logistic Regression uses gradient descent to optimise the log-likelihood. If Age ranges from 18-60 and EstimatedSalary from 15K-150K, the gradients for the salary coefficient will be tiny compared to age. Training becomes slow and convergence is sensitive to the learning rate.

After StandardScaler, both features have mean 0 and std 1 — equal gradient magnitudes, faster convergence.

In [ ]:
# Split the data into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print("Data split completed:")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

# Feature scaling (CRITICAL for logistic regression)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed:")
print(f"Original feature ranges:")
print(f"- Age: {X_train[:, 0].min():.0f} to {X_train[:, 0].max():.0f}")
print(f"- Salary: ${X_train[:, 1].min():,.0f} to ${X_train[:, 1].max():,.0f}")

print(f"\nScaled feature ranges:")
print(f"- Age: {X_train_scaled[:, 0].min():.2f} to {X_train_scaled[:, 0].max():.2f}")
print(f"- Salary: {X_train_scaled[:, 1].min():.2f} to {X_train_scaled[:, 1].max():.2f}")

print("\nData is ready for logistic regression training!")

In [ ]:
print(x_train)

In [ ]:
print(y_train)

In [ ]:
print(x_test)

In [ ]:
print(y_test)

## Step 6: Apply Feature Scaling

We fit the `StandardScaler` on the training data only, then transform both sets.

The output shows Age and Salary now expressed as standard deviations from the training mean — both features on the same scale. A value of 2.0 means "2 standard deviations above the training mean", regardless of whether the original unit was years or dollars.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
print(x_train)

In [ ]:
print(x_test)

## Step 7: Train the Logistic Regression Model

Logistic Regression fits a sigmoid curve to the training data:

$$P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 \cdot \text{Age} + \beta_2 \cdot \text{Salary})}}$$

Training finds the coefficients β that maximise the likelihood of the observed outcomes (maximum likelihood estimation, solved via gradient descent).

**`random_state=0`** — Logistic Regression with the default `lbfgs` solver is deterministic. The `random_state` parameter does not affect the result here, but is included for reproducibility if you later switch to a stochastic solver.

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(x_train, y_train)

## Step 8: Make Predictions

Two prediction types:

**Single observation** — `[[30, 87000]]` (30-year-old, £87k salary). The double brackets create the 2D array shape `(1, 2)` that sklearn expects.

**Note:** The input must be scaled with the same scaler used during training before passing to `predict()`. The code calls `sc.transform([[30, 87000]])` — this applies the training-set mean and std to the new point.

**All test predictions** — `y_pred` contains 100 predictions. The side-by-side comparison with `y_test` lets you inspect individual misclassifications before looking at summary metrics.

In [ ]:
print(classifier.predict(sc.transform([[30,87000]])))

In [ ]:
y_pred = classifier.predict(x_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

## Step 9: Evaluate the Model

Four metrics together give a complete picture:

| Metric | Formula | What it answers |
|--------|---------|----------------|
| **Accuracy** | (TP + TN) / total | What fraction of all predictions were correct? |
| **Precision** | TP / (TP + FP) | Of all predicted buyers, how many actually bought? |
| **Recall** | TP / (TP + FN) | Of all actual buyers, how many did we catch? |
| **F1** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall |

**Which metric to prioritise?** It is a business decision:
- Maximise **recall** when missing a buyer is costly (you would rather send an extra ad than miss a sale)
- Maximise **precision** when false positives are costly (e.g., an expensive personal sales call to a non-buyer)
- Use **F1** when you need a single balanced number

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
print(f'Precision: {precision_score(y_test, y_pred):.2f}')
print(f'Recall: {recall_score(y_test, y_pred):.2f}')
print(f'F1-score: {f1_score(y_test, y_pred):.2f}')
print(f'ROC-AUC: {roc_auc_score(y_test, classifier.predict_proba(x_test)[:,1]):.2f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## Step 10: Visualise the Training Set Decision Boundary

The plot shows two regions:
- **Red region** — where the model predicts class 0 (did not purchase)
- **Green region** — where the model predicts class 1 (purchased)

The straight diagonal line separating the two regions is the **decision boundary** — the set of all points where P(y=1) = 0.5. Logistic Regression always produces a linear boundary in feature space.

**What to check:** Training accuracy looks good by design — the model was fit to these points. A few red dots in the green region (and vice versa) indicate either overlapping classes or the linear boundary not being expressive enough for this data.

In [ ]:
from matplotlib.colors import ListedColormap
# 1. Get the training data (same as original)
X_set, y_set = sc.inverse_transform(x_train), y_train

# 2. Calculate grid parameters that match original visual density but are memory-safe
# Original had step=0.25 for age (very dense) and step=0.25 for salary (extremely dense)
# We'll use linspace to get similar visual quality with controlled memory usage

# Calculate equivalent number of points to match original step size
age_points = int((X_set[:, 0].max() + 10 - (X_set[:, 0].min() - 10)) / 0.25)
salary_points = int((X_set[:, 1].max() + 1000 - (X_set[:, 1].min() - 1000)) / 0.25)

# Cap at reasonable limits to prevent memory errors
max_points = 500  # Adjust based on your system's memory
age_points = min(age_points, max_points)
salary_points = min(salary_points, max_points)

# 3. Create the grid (same range as original but safer implementation)
X1, X2 = np.meshgrid(
    np.linspace(X_set[:, 0].min() - 10, X_set[:, 0].max() + 10, age_points),
    np.linspace(X_set[:, 1].min() - 1000, X_set[:, 1].max() + 1000, salary_points)
)

# 4. Predict and plot (same as original but with memory protection)
try:
    # Same prediction logic as original
    Z = classifier.predict(sc.transform(np.array([X1.ravel(), X2.ravel()]).T))
    Z = Z.reshape(X1.shape)
    
    # Create plot with original styling
    plt.figure(figsize=(8, 6))  # Slightly larger than default for better visibility
    plt.contourf(X1, X2, Z, alpha=0.75, cmap=ListedColormap(('red', 'green')))
    
    # Plot training points with original colors and no edgecolors
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                    c=ListedColormap(('red', 'green'))(i), label=j)
    
    # Original title and labels
    plt.xlim(X1.min(), X1.max())
    plt.ylim(X2.min(), X2.max())
    plt.title('Logistic Regression (Training set)')
    plt.xlabel('Age')
    plt.ylabel('Estimated Salary')
    plt.legend()
    plt.show()

except MemoryError:
    # Fallback option if even reduced grid is too large
    print("Memory error - using simplified visualization")
    plt.figure(figsize=(8, 6))
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                    c=ListedColormap(('red', 'green'))(i), label=j)
    plt.title('Training Data (Decision Boundary Too Memory Intensive)')
    plt.xlabel('Age')
    plt.ylabel('Estimated Salary')
    plt.legend()
    plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = sc.inverse_transform(x_train), y_train
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 10, stop = X_set[:, 0].max() + 10, step = 0.25),
                     np.arange(start = X_set[:, 1].min() - 1000, stop = X_set[:, 1].max() + 1000, step = 0.25))
plt.contourf(X1, X2, classifier.predict(sc.transform(np.array([X1.ravel(), X2.ravel()]).T)).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1], c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Logistic Regression (Training set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()

## Step 11: Visualise the Test Set Decision Boundary

The same decision boundary from training, now evaluated against the 100 held-out test customers.

**What to check:**
- If the test plot looks as clean as the training plot → good generalisation, the linear boundary captures the true pattern
- If the test plot has significantly more misclassifications → the model may be underfitting (the true boundary is nonlinear)

For Social Network Ads data, Logistic Regression typically achieves ~85% test accuracy. The cluster of younger high-salary users who do not purchase (bottom-right of the bought region) is where a linear boundary consistently struggles — a kernel SVM or polynomial features would separate these better.

In [ ]:
from matplotlib.colors import ListedColormap
# 1. Get the test data
X_set, y_set = sc.inverse_transform(x_test), y_test

# 2. Calculate equivalent grid density with memory protection
# Original had step=0.25 for both axes (extremely dense)
# We'll calculate equivalent point count but cap it

# Calculate how many points the original would generate
age_points = int((X_set[:, 0].max() + 10 - (X_set[:, 0].min() - 10)) / 0.25)
salary_points = int((X_set[:, 1].max() + 1000 - (X_set[:, 1].min() - 1000)) / 0.25)

# Set safe maximum points (adjust based on your system memory)
max_points = 500
age_points = min(age_points, max_points)
salary_points = min(salary_points, max_points)

# 3. Create the grid with identical range but safer method
X1, X2 = np.meshgrid(
    np.linspace(X_set[:, 0].min() - 10, X_set[:, 0].max() + 10, age_points),
    np.linspace(X_set[:, 1].min() - 1000, X_set[:, 1].max() + 1000, salary_points)
)

# 4. Predict and plot with original styling
try:
    # Same prediction logic as original
    Z = classifier.predict(sc.transform(np.column_stack((X1.ravel(), X2.ravel()))))
    Z = Z.reshape(X1.shape)
    
    # Create plot with original test set styling
    plt.figure(figsize=(8, 6))
    plt.contourf(X1, X2, Z, alpha=0.75, cmap=ListedColormap(('red', 'green')))
    
    # Plot test points with original colors
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                   c=ListedColormap(('red', 'green'))(i), label=j)
    
    # Original axis limits and labels
    plt.xlim(X1.min(), X1.max())
    plt.ylim(X2.min(), X2.max())
    plt.title('Logistic Regression (Test set)')
    plt.xlabel('Age')
    plt.ylabel('Estimated Salary')
    plt.legend()
    plt.show()

except MemoryError:
    # Fallback visualization if memory is still insufficient
    print("Note: Using simplified visualization due to memory constraints")
    plt.figure(figsize=(8, 6))
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                   c=ListedColormap(('red', 'green'))(i), label=j)
    plt.title('Test Set Data (Simplified)')
    plt.xlabel('Age')
    plt.ylabel('Estimated Salary')
    plt.legend()
    plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = sc.inverse_transform(x_test), y_test
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 10, stop = X_set[:, 0].max() + 10, step = 0.25),
                     np.arange(start = X_set[:, 1].min() - 1000, stop = X_set[:, 1].max() + 1000, step = 0.25))
plt.contourf(X1, X2, classifier.predict(sc.transform(np.array([X1.ravel(), X2.ravel()]).T)).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1], c = ListedColormap(('red', 'green'))(i), label = j)
plt.title('Logistic Regression (Test set)')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.legend()
plt.show()

## Step 12: Summary and Interpretation

The metrics table gives a quick diagnostic read:

- **Accuracy ~85%** — strong for a linear model on this data
- **Precision vs Recall trade-off** — check whether FP or FN dominate the error; this guides whether to adjust the classification threshold

**When to choose Logistic Regression:**
- Baseline for any binary classification problem — fast, interpretable, well-calibrated probabilities
- When you need to explain model decisions to non-technical stakeholders (the coefficients directly show feature importance)
- When the decision boundary is approximately linear

**When to move to another model:**
- Nonlinear boundary needed → try SVM with RBF kernel, Decision Tree, or Random Forest
- Many features with complex interactions → gradient boosting (XGBoost, LightGBM)

In [ ]:
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, classifier.predict_proba(x_test)[:,1])
}
for k, v in metrics.items():
    print(f'{k}: {v:.2f}')

# If all metrics are above 0.7, model is generally good.
# If metrics are low, check for class imbalance, feature selection, or try other models.

# Check for class imbalance
print('Class distribution:', np.bincount(y_test))

# Show coefficients for interpretation
print('Feature coefficients:', classifier.coef_)
print('Intercept:', classifier.intercept_)